# Analyse de donnees

- **Objectif :** explorer le jeu de données afin de valider la pertinence des variables statiques et démographiques avant la phase de modélisation prédictive.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from config import PROCESSED_DATA
from scipy.stats import mannwhitneyu, chi2_contingency, false_discovery_control
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

## Preparation des donnees

1. Chargement des donnees

In [ ]:
data = pd.read_parquet(PROCESSED_DATA / 'longitudinal_panel.parquet')
data.head()

2. Exclusion des variables redendantes

In [ ]:
redundant_cols = [
    'id_inscription', 'id_etudiant', 'id_programme',
    'resultat_final', 'date_debut', 'date_annulation',
    'date_debut_programme', 'cancel_day', 'churn'
]

data.drop(columns=redundant_cols, inplace=True, errors='ignore')
data.head()

3. Type des variables

In [ ]:
num = [
    'age',
    'nb_inscriptions_precedentes',
    'volume_per_active_day_30d',
    'days_since_last_activity',
    'active_days_diff',
    'n_evaluations_30d',
    'avg_note_cum',
    'avg_lateness_cum'
]

cat = [
    'niveau_scolaire',
    'annee_scolaire',
    'semestre',
    'genre',
    'region',
    'situation_handicap'
]

churn = 'target_next_30d'

## Visualisation des donnees 

In [ ]:
print(data[churn].value_counts(normalize=True))

print(
    pd.crosstab(
        data['checkpoint_day'],
        data[churn],
        normalize='index'
    )
)

sns.barplot(data=data, x='checkpoint_day', y=churn, errorbar=None)
plt.ylabel("Taux d'abandon dans les 30 jours")
plt.show()

1. Les variables numeriques

In [ ]:
data[num].describe()

In [ ]:
data[num].isna().mean().sort_values(ascending=False)

In [ ]:
for col in num:
    sns.boxplot(data=data, x=churn, y=col, showfliers=False)
    plt.title(col)
    plt.show()

2. Les variables Categoriques

In [ ]:
print(data[churn].value_counts(normalize=True))
for col in cat:
    print(f'Pour la variable {col}:')
    print(data[col].value_counts())
    print(pd.crosstab(data[col],data[churn],normalize='index'))
    print('\n')


In [ ]:
# visualisation
base_rate = data[churn].mean()

for col in cat:
    churn_rate = data.groupby(col, observed=True)[churn].mean().sort_values(ascending=False)

    sns.barplot(x=churn_rate.values, y=churn_rate.index.astype(str), orient='h', color='steelblue')
    plt.axvline(base_rate, color='red', linestyle='--', label=f'Taux global ({base_rate:.1%})')
    plt.xlabel("Taux d'abandon")
    plt.ylabel(col)
    plt.title(f"Taux d'abandon par {col}")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Significativite des variables

1. Test de Mann-Whitney pour les variables numeriques

> **$H_0$:** la distribution de la variable est identique entre les etudiants qui abandonnent (churn=1) et ceux qui n'abandonnent pas (churn=0).

> **$H_1$:** les deux distributions different (l'un des deux groupes tend a avoir des valeurs plus elevees que l'autre).

In [ ]:
res = []

for col in num:
    group0 = data.loc[data[churn]==0, col].dropna()
    group1 = data.loc[data[churn]==1, col].dropna()
    n0, n1 = len(group0), len(group1)

    U, p = mannwhitneyu(group0, group1, alternative='two-sided')
    score = U / (n0*n1)

    res.append({
        'variable': col,
        'test': 'Mann-Whitney',
        'statistique': U,
        'p_value': p,
        'score': score
    })

num_results = pd.DataFrame(res)
num_results

2. Test de Chi-2 pour les variables categoriques

> **$H_0$:** la variable et churn sont independants (la proportion de churn est la meme dans toutes les categories de la variable).

> **$H_1$:** la variable et churn sont associes (la proportion de churn differe selon la categorie).

In [ ]:
def cramersV(contingency_table):
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    n = contingency_table.values.sum()
    min_dim = min(contingency_table.shape) - 1
    v = np.sqrt(chi2 / (n*min_dim))
    return chi2, p, v

cat_res = []

for col in cat:
    table = pd.crosstab(data[col], data[churn])
    chi2, p, v = cramersV(table)

    cat_res.append({
        'variable': col,
        'test': 'Chi-2',
        'statistique': chi2,
        'p_value': p,
        'score': v
    })

cat_results = pd.DataFrame(cat_res)
cat_results

3. Correction de Benjamini-Hochberg

In [ ]:
# Ajustement des p_values
results = pd.concat([num_results, cat_results], ignore_index=True)
results['p_value_adj'] = false_discovery_control(results['p_value'], method='bh')
results.sort_values('p_value_adj')

## Correlation

1. Pour les variables numeriques

In [ ]:
corr = data[num].corr(method='spearman')
corr

In [ ]:
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.show()

2. Pour les variables categoriques

In [ ]:
cramers_matrix = pd.DataFrame(index=cat, columns=cat, dtype=float)
for col1 in cat:
    for col2 in cat:
        table = pd.crosstab(data[col1], data[col2])
        _, _, v = cramersV(table)
        cramers_matrix.loc[col1, col2] = v

cramers_matrix = cramers_matrix.astype(float)
sns.heatmap(cramers_matrix, annot=True, cmap='coolwarm', vmin=0, vmax=1)
plt.show()

## Etude de multicolinéarité

In [ ]:
vif_data = data[num].replace([np.inf, -np.inf], np.nan).dropna()
vif_data = vif_data.loc[:, vif_data.nunique() > 1]

vif_design = add_constant(vif_data.astype(float))

vif_results = pd.DataFrame({
    'variable': vif_design.columns,
    'VIF': [
        variance_inflation_factor(vif_design.values, i)
        for i in range(vif_design.shape[1])
    ]
})

vif_results = vif_results[vif_results['variable'] != 'const'].sort_values(
    'VIF', ascending=False
).reset_index(drop=True)

vif_results

## Conclusion